# VERA — Full Ablation Study (Language Table, 6 conditions × 3 seeds)

### Root causes of VERA ≈ BC (all fixed in this version)
| Root cause | Fix applied |
|---|---|
| Alignment loss: zero gradient (frozen CLIP embeddings both sides) | `alignment_loss_coef` → **0.0** |
| Frozen CLIP: identical features for all 8 push directions | `unfreeze_clip_vision=True`, `clip_vision_lr=1e-5` |
| LR too low → barely-moving CE loss | LR 2e-4 → **5e-4** (backbone) |
| Batch too small (2 examples/class) → noisy CE | batch_size 16 → **32** |
| Reg loss (0.5) swamped CE gradient early | reg_coef 0.5 → **0.1** |

### Ablation conditions (6 core variants)
| # | Name | Streams removed |
|---|---|---|
| 0 | Full VERA | — (all 5 streams) |
| 1 | BC baseline | E_act, E_exp, hist TF |
| 2 | No language feedback | E_act, E_exp (hist TF + gate kept) |
| 3 | No E_exp only | E_exp / Stream 3b |
| 4 | No E_act only | E_act / Stream 3a |
| 5 | No history TF | history sub-transformer |

**Expected on A100:** ~6 × 3 seeds × ~25 min = ~7–8 hours.


In [ ]:
# ── Cell 1: GPU + RAM check ──────────────────────────────────────────────────
# ⚠️  This notebook requires a CUDA GPU (T4 or A100).
#    If you are on a TPU runtime: Runtime → Change runtime type → T4 GPU or A100 GPU
#    TPU runtimes do NOT have CUDA and the trainer will crash with
#    "Torch not compiled with CUDA enabled".

import torch, psutil, os

cuda_ok = torch.cuda.is_available()
print(f'CUDA GPU : {cuda_ok}')
if cuda_ok:
    print(f'  Device : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print()
    print('  ❌  No CUDA GPU found.')
    print('  → Runtime → Change runtime type → T4 GPU  (or A100 for ~2× speed)')
    print('  → Then re-run all cells from the top.')
    print()
    print('  Common cause: you are on a TPU v6e-1 runtime.')
    print('  TPU requires torch_xla and is NOT supported by this notebook.')
    raise SystemExit('Switch to a GPU runtime and re-run.')

print(f'RAM  : {psutil.virtual_memory().total/1e9:.1f} GB total  |  '
      f'{psutil.virtual_memory().available/1e9:.1f} GB free')
print('GPU check passed ✓')

In [ ]:
# ── Cell 2: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted:', DRIVE_ROOT)

In [ ]:
# ── Cell 3: Install deps ─────────────────────────────────────────────────────
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex
import clip, torch, yaml
print('torch:', torch.__version__, '  CLIP ready ✓')

In [ ]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os
REPO_URL = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR = '/content/RLConditionedVLA'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo:', os.getcwd())

In [ ]:
# ── Cell 5: Generate rendered synthetic data ─────────────────────────────────
# Write to LOCAL disk (/content/) first — avoids Drive disconnect on bulk writes.
# Then tar → copy single file to Drive for persistence across sessions.
#
# Visual content per episode:
#   64×64 canvas, red circle=block, green circle=goal, light-gray background
#   Action = normalised block→goal vector  (true directional signal)
#   Instruction = "push the block to the right/upward/..." (direction-encoded)
#   Block moves 4 px/step → frames change per timestep
#
# Memory: 900 eps × 20 steps × 64×64×3 ≈ 221 MB local disk

import os, pickle, tarfile, shutil
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

# ── paths ─────────────────────────────────────────────────────────────────────
LOCAL_SYNTH   = Path('/content/lt_rendered_synth')      # ← local disk (fast, no disconnect)
SYNTH_TAR     = f'{DRIVE_ROOT}/lt_rendered_synth.tar'  # ← single tar on Drive
TRAIN_DIR     = LOCAL_SYNTH / 'train'
VAL_DIR       = LOCAL_SYNTH / 'val'
IMG_STORE     = 64
TRAIN_N       = 900
VAL_N         = 135
EP_LEN        = 20

DIR_NAMES = [
    "to the right", "up and to the right", "upward", "up and to the left",
    "to the left",  "down and to the left", "downward", "down and to the right",
]

def count_ep(d):
    return len(list(Path(d).glob('episode_*'))) if Path(d).exists() else 0

DATA_SOURCE = 'unknown'

# ── restore from Drive tar if local copy missing ───────────────────────────────
if count_ep(TRAIN_DIR) >= 200:
    DATA_SOURCE = 'local_cached'
    print(f'Local cache OK: train={count_ep(TRAIN_DIR)}  val={count_ep(VAL_DIR)}')

elif os.path.exists(SYNTH_TAR):
    DATA_SOURCE = 'drive_tar'
    print('Restoring from Drive tar to /content/ ...')
    LOCAL_SYNTH.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(SYNTH_TAR, 'r') as t:
        t.extractall('/content/')
    print(f'Restored: train={count_ep(TRAIN_DIR)}  val={count_ep(VAL_DIR)}')

else:
    # ── render episodes → local disk ─────────────────────────────────────────
    def render_frame(bx, by, gx, gy, size=IMG_STORE):
        img = Image.new('RGB', (size, size), (235, 235, 235))
        d   = ImageDraw.Draw(img)
        d.ellipse([gx-6, gy-6, gx+6, gy+6],  fill=(30, 180, 30))   # goal (green)
        d.ellipse([int(bx)-8, int(by)-8, int(bx)+8, int(by)+8], fill=(210, 40, 40))  # block (red)
        return np.array(img, dtype=np.uint8)

    def make_ep(ep_idx):
        T      = EP_LEN + np.random.randint(-3, 6)
        bin_id = ep_idx % 8
        instr  = f"push the block {DIR_NAMES[bin_id]}"
        mg     = 20
        bx = float(np.random.randint(mg, IMG_STORE - mg))
        by = float(np.random.randint(mg, IMG_STORE - mg))
        angle  = bin_id * (2 * np.pi / 8)
        spread = np.random.uniform(20, 28)
        gx = float(np.clip(bx + np.cos(angle) * spread, mg, IMG_STORE - mg))
        gy = float(np.clip(by + np.sin(angle) * spread, mg, IMG_STORE - mg))
        steps = []
        for t in range(T):
            frame = render_frame(bx, by, gx, gy)
            dx = gx - bx;  dy = gy - by
            dist = float(np.sqrt(dx*dx + dy*dy)) + 1e-6
            act = np.clip(np.array([dx/dist * 0.8 + np.random.normal(0, 0.04),
                                    dy/dist * 0.8 + np.random.normal(0, 0.04)],
                                   dtype=np.float32), -1.0, 1.0)
            rew = float(min(1.0, max(0.0, 1.0 - dist / (IMG_STORE * 0.5))))
            steps.append({'obs': {'rgb': frame}, 'action': act,
                          'reward': rew, 'instruction': instr})
            step_px = 4.0
            bx = float(np.clip(bx + dx/dist*step_px + np.random.normal(0, 0.5),
                               mg//2, IMG_STORE - mg//2))
            by = float(np.clip(by + dy/dist*step_px + np.random.normal(0, 0.5),
                               mg//2, IMG_STORE - mg//2))
        return steps

    def save_split(n, start_idx, out_dir):
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        for i in tqdm(range(n), desc=str(Path(out_dir).name), leave=False):
            steps = make_ep(start_idx + i)
            ep_d  = Path(out_dir) / f'episode_{i:05d}'
            ep_d.mkdir(exist_ok=True)
            with open(ep_d / 'steps.pkl', 'wb') as f:   # local disk write — no Drive!
                pickle.dump(steps, f)

    print(f'Generating {TRAIN_N}+{VAL_N} rendered episodes to local disk...')
    save_split(TRAIN_N, 0,       TRAIN_DIR)
    save_split(VAL_N,   TRAIN_N, VAL_DIR)
    DATA_SOURCE = 'rendered_synthetic'

    # ── tar local dir → single Drive write ───────────────────────────────────
    print('Saving tar to Drive (single write, no disconnect risk)...')
    all_files = [p for p in LOCAL_SYNTH.rglob('*') if p.is_file()]
    LOCAL_TAR  = '/content/lt_rendered_tmp.tar'
    with tarfile.open(LOCAL_TAR, 'w') as t:
        for p in tqdm(all_files, desc='pack', leave=False):
            t.add(str(p), arcname=str(p.relative_to(LOCAL_SYNTH.parent)))
    shutil.copy2(LOCAL_TAR, SYNTH_TAR)   # single Drive write
    os.remove(LOCAL_TAR)
    print(f'Tar saved to Drive ✓  ({os.path.getsize(SYNTH_TAR)/1e6:.0f} MB)')

import psutil, pickle as _pkl
print(f'\nData source     : {DATA_SOURCE}')
print(f'Train episodes  : {count_ep(TRAIN_DIR)}')
print(f'Val episodes    : {count_ep(VAL_DIR)}')
print(f'RAM used        : {psutil.virtual_memory().used/1e9:.1f} GB')
assert count_ep(TRAIN_DIR) >= 100

ep0 = _pkl.load(open(sorted(TRAIN_DIR.glob("episode_*/steps.pkl"))[0], "rb"))
act0 = np.array([s["action"] for s in ep0], dtype=np.float32)
print(f'Sample instr    : {ep0[0]["instruction"]}')
print(f'Sample actions  : mean=[{act0[:,0].mean():.2f},{act0[:,1].mean():.2f}] std=[{act0[:,0].std():.2f},{act0[:,1].std():.2f}]')
print(f'Sample frame    : shape={ep0[0]["obs"]["rgb"].shape}  dtype={ep0[0]["obs"]["rgb"].dtype}')
print('Done ✓')


In [ ]:
# ── Cell 6: Smoke test ───────────────────────────────────────────────────────
from data.trajectory_dataset import load_language_table
import psutil

eps = load_language_table(f'{LT_DATA_PATH}/train')
assert len(eps) > 0, 'No episodes loaded'
e0 = eps[0]
print(f'Episodes      : {len(eps)}')
print(f'Frames shape  : {e0["frames"].shape}   (stored size — resized to 224 at batch time)')
print(f'Actions shape : {e0["actions"].shape}   (8-bin discrete)')
print(f'ActionV shape : {e0["action_vectors"].shape}  (expect T×2 continuous)')
print(f'Instruction   : {e0["instruction"]}')
print(f'Action range  : [{e0["actions"].min()}, {e0["actions"].max()}]  (8 bins expected)')
print(f'RAM used      : {psutil.virtual_memory().used/1e9:.1f} GB')
print('Loader OK ✓')

In [ ]:
# ── Cell 7: Base config ───────────────────────────────────────────────────────
import yaml

with open(f'{REPO_DIR}/configs/config.yaml') as f:
    base_cfg = yaml.safe_load(f)

# ── Dataset: rendered synthetic (64px stored → 224 at batch time) ─────────────
# SYNTH_PATH is defined in Cell 5. Using rendered synthetic, NOT real Drive data.
base_cfg['data']['episodes_path'] = str(TRAIN_DIR)
base_cfg['data']['dataset_type']  = 'language_table'
base_cfg['data']['img_size']      = 224

# ── Model ─────────────────────────────────────────────────────────────────────
base_cfg['model']['num_actions']          = 8
base_cfg['model']['action_dim']           = 2
base_cfg['model']['freeze_clip']          = True   # text encoder frozen
base_cfg['model']['unfreeze_clip_vision'] = False  # keep vision frozen too
                                                     # (rendered images are simple;
                                                     # frozen CLIP CAN distinguish
                                                     # red vs green circles by color)

# ── LT action vocabulary (9 entries: 0-7 + 8=padding) ────────────────────────
base_cfg['vera']['action_vocab'] = {
    0: 'I pushed the object to the right',
    1: 'I pushed the object up and to the right',
    2: 'I pushed the object upward',
    3: 'I pushed the object up and to the left',
    4: 'I pushed the object to the left',
    5: 'I pushed the object down and to the left',
    6: 'I pushed the object downward',
    7: 'I pushed the object down and to the right',
    8: 'no previous action taken',
}

# ── Loss coefficients ─────────────────────────────────────────────────────────
# alignment_loss_coef=0.0: InfoNCE was computed on frozen CLIP embeddings → zero grad
base_cfg['vera']['alignment_loss_coef']  = 0.0
base_cfg['vera']['regression_loss_coef'] = 0.3   # helps share direction signal

# ── Training ──────────────────────────────────────────────────────────────────
base_cfg['training'].update({
    'output_dir'             : f'{DRIVE_ROOT}/checkpoints',
    'device'                 : 'auto',
    'epochs'                 : 80,
    'lr'                    : 3e-4,
    'lr_min'                : 1e-6,
    'batch_size'            : 64,    # larger batch → cleaner CE gradient over 8 classes
    'num_workers'           : 0,
    'val_fraction'          : 0.15,
    'label_smoothing'       : 0.05,
    'grad_clip'             : 1.0,
    'weight_decay'          : 1e-4,
    'save_every'            : 25,
    'early_stopping_patience': 20,
})

lt_cfg_path = f'{REPO_DIR}/configs/lt_colab_runtime.yaml'
with open(lt_cfg_path, 'w') as f:
    yaml.dump(base_cfg, f)

print('Config written ✓')
print(f'  episodes_path        : {base_cfg["data"]["episodes_path"]}')
print(f'  dataset_type         : language_table (rendered synthetic)')
print(f'  unfreeze_clip_vision : {base_cfg["model"]["unfreeze_clip_vision"]}  (frozen is fine — synthetic has clear colors)')
print(f'  alignment_loss_coef  : {base_cfg["vera"]["alignment_loss_coef"]}')
print(f'  regression_loss_coef : {base_cfg["vera"]["regression_loss_coef"]}')
print(f'  lr                   : {base_cfg["training"]["lr"]}')
print(f'  batch_size           : {base_cfg["training"]["batch_size"]}')
print(f'  vocab entries        : {len(base_cfg["vera"]["action_vocab"])}')


In [ ]:
# ── Cell 8: Run 6 conditions × 3 seeds ───────────────────────────────────────
#
# Conditions:
#  0  full_vera    — all 5 streams (the full model)
#  1  bc_baseline  — vision + instruction only (no E_act, no E_exp, no hist TF)
#  2  no_lang      — removes E_act + E_exp; keeps hist TF + reward gate
#  3  no_exp       — removes E_exp (Stream 3b) only; keeps E_act
#  4  no_act       — removes E_act (Stream 3a) only; keeps E_exp
#  5  no_hist_tf   — removes history sub-transformer; keeps language streams
#
# Results saved to Drive after every seed. Re-run safe (done seeds skipped).

import copy, gc, json as _json
import numpy as np, torch, yaml, psutil
from training.sft_trainer_vera import sft_train

SEEDS = [42, 123, 456]

CONDITIONS = [
    {
        'name': 'full_vera',
        'display': 'Full VERA (all 5 streams)',
        'overrides': {},
    },
    {
        'name': 'bc_baseline',
        'display': 'BC baseline (no lang, no hist TF)',
        'overrides': {
            ('vera','use_lang_feedback'):     False,
            ('vera','use_consequence_token'): False,
            ('vera','use_temporal_history'):  False,
        },
    },
    {
        'name': 'no_lang',
        'display': 'No language feedback (keeps hist TF + gate)',
        'overrides': {
            ('vera','use_lang_feedback'):     False,
            ('vera','use_consequence_token'): False,
        },
    },
    {
        'name': 'no_exp',
        'display': 'No E_exp / Stream 3b (action narration only)',
        'overrides': {
            ('vera','use_consequence_token'): False,
        },
    },
    {
        'name': 'no_act',
        'display': 'No E_act / Stream 3a (experience token only)',
        'overrides': {
            ('vera','use_lang_feedback'): False,
        },
    },
    {
        'name': 'no_hist_tf',
        'display': 'No history sub-transformer (flat positional history)',
        'overrides': {
            ('vera','use_temporal_history'): False,
        },
    },
]

with open(lt_cfg_path) as f:
    base_cfg = yaml.safe_load(f)

all_results = {}

for ci, cond in enumerate(CONDITIONS):
    cond_results = {}
    print(f'\n{"═"*62}')
    print(f'  [{ci+1}/{len(CONDITIONS)}] {cond["display"]}')
    print(f'{"═"*62}')

    for seed in SEEDS:
        run_dir  = f'{DRIVE_ROOT}/checkpoints/lt_{cond["name"]}/seed{seed}'
        log_path = f'{run_dir}/sft_vera_log.json'

        # ── Skip if already done ──────────────────────────────────────────────
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            if log:
                best = max(r['val_acc'] for r in log)
                print(f'  [SKIP] seed={seed}  best val_acc={best:.4f}')
                cond_results[f'seed{seed}'] = best
                continue

        ram_gb = psutil.virtual_memory().used / 1e9
        print(f'\n  seed={seed}  RAM={ram_gb:.1f} GB')
        os.makedirs(run_dir, exist_ok=True)

        cfg = copy.deepcopy(base_cfg)
        for (sec, key), val in cond['overrides'].items():
            cfg[sec][key] = val
        cfg['training']['seed']       = seed
        cfg['training']['output_dir'] = run_dir

        torch.manual_seed(seed)
        np.random.seed(seed)
        sft_train(cfg)

        # Free memory between runs
        torch.cuda.empty_cache()
        gc.collect()

        best = 0.0
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            best = max(r['val_acc'] for r in log) if log else 0.0
        cond_results[f'seed{seed}'] = best
        print(f'  ✓ seed={seed}  best={best:.4f}')

    all_results[cond['name']] = cond_results

summary = {'data_source': DATA_SOURCE, 'results': all_results}
_json.dump(summary, open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json','w'), indent=2)
print(f'\n✓ All conditions done.')

In [ ]:
# ── Cell 9: Paper table ───────────────────────────────────────────────────────
import json, numpy as np

data = json.load(open(f'{DRIVE_ROOT}/checkpoints/lt_summary.json'))
res, src = data['results'], data.get('data_source','?')

ORDER = [
    ('bc_baseline', 'BC/SFT baseline             (no lang, no hist TF)'),
    ('no_lang',     'No lang feedback†            (hist TF + gate kept)'),
    ('no_exp',      'No E_exp / Stream 3b         (act narration only)'),
    ('no_act',      'No E_act / Stream 3a         (experience only)'),
    ('no_hist_tf',  'No history TF                (flat positional hist)'),
    ('full_vera',   'Full VERA ★                  (all 5 streams)'),
]

print(f'\nData source: {src}')
print('─'*80)
print('Language-Table ablation — validation action-classification accuracy')
print(f'{"Variant":<52} {"S42":>6} {"S123":>6} {"S456":>6} {"Mean±Std":>12}')
print('─'*80)

vera_mu = None
for key, display in ORDER:
    if key not in res:
        print(f'  {display:<52}  (not run)')
        continue
    v = [res[key].get(f'seed{s}', float('nan')) for s in [42,123,456]]
    mu, std = float(np.nanmean(v)), float(np.nanstd(v))
    if key == 'full_vera': vera_mu = mu
    vals = '  '.join(f'{x*100:5.1f}' if not np.isnan(x) else '  ---' for x in v)
    marker = ' ←best' if key == 'full_vera' else ''
    print(f'  {display:<52}  {vals}   {mu*100:.1f}±{std*100:.1f}{marker}')

print('─'*80)

# Delta over BC
if 'bc_baseline' in res and vera_mu is not None:
    bc_v = [res['bc_baseline'].get(f'seed{s}', float('nan')) for s in [42,123,456]]
    bc_mu = float(np.nanmean(bc_v))
    delta = (vera_mu - bc_mu) * 100
    sign  = '+' if delta >= 0 else ''
    print(f'  VERA vs BC delta: {sign}{delta:.2f} pp')

print()
print('† Removes E_act (Stream 3a) and E_exp (Stream 3b); hist TF and reward gate remain.')
print('\nPaste into vera_neurips.tex → tab:ablations')

In [ ]:
# ── Cell 10: Download results ─────────────────────────────────────────────────
from google.colab import files
import shutil
shutil.make_archive('/content/lt_results','zip', f'{DRIVE_ROOT}/checkpoints')
files.download('/content/lt_results.zip')
print('Download started')